In [0]:
--NULL CHECK

INSERT INTO dq.data_quality_results
SELECT
    'crm_sales_details' AS table_name,
    'null_order_number' AS check_name,
    COUNT(*) FILTER (WHERE sls_ord_num IS NULL) AS failed_count,
    COUNT(*) AS total_count,
    try_divide(COUNT(*), COUNT(*)) AS failure_percentage,
    CASE 
        WHEN COUNT(*) FILTER (WHERE sls_ord_num IS NULL) > 0 THEN 'FAIL'
        ELSE 'PASS'
    END,
    current_timestamp()
FROM silver.crm_sales_details;

In [0]:
--DUPLICATE CHECK
INSERT INTO dq.data_quality_results
SELECT
    'crm_sales_details',
    'duplicate_orders',
    COUNT(*) AS failed_count,
    COUNT(*) AS total_count,
    try_divide(COUNT(*), COUNT(*)) AS failure_percentage,
     CASE 
        WHEN COUNT(*) > 0 THEN 'FAIL'
        ELSE 'PASS'
    END,
    current_timestamp()
FROM (
    SELECT sls_ord_num, sls_prd_key,COUNT(*)
    FROM silver.crm_sales_details
    GROUP BY sls_ord_num, sls_prd_key
    HAVING COUNT(*) > 1
);

In [0]:
--JOIN CHECK
INSERT INTO dq.data_quality_results
SELECT
    'crm_sales_details',
    'missing_product_key',
    COUNT(*) AS failed_count,
    COUNT(*) AS total_count,
   try_divide(COUNT(*), COUNT(*)) AS failure_percentage,
    CASE 
        WHEN COUNT(*) > 0 THEN 'FAIL'
        ELSE 'PASS'
    END,
    current_timestamp()
FROM silver.crm_sales_details sd
LEFT JOIN gold.dim_product pr
    ON sd.sls_prd_key = pr.product_number
    --AND sd.sls_order_dt >= pr.start_date
   -- AND (sd.sls_order_dt <= pr.end_date OR pr.end_date IS NULL)
WHERE pr.product_key IS NULL;

In [0]:
--CUSTOMER JOIN CHECK
INSERT INTO dq.data_quality_results
SELECT
    'crm_sales_details',
    'missing_customer_key',
    COUNT(*) AS failed_count,
    COUNT(*) AS total_count,
    TRY_DIVIDE(COUNT(*) , COUNT(*)) AS failure_percentage,
    CASE 
        WHEN COUNT(*) > 0 THEN 'FAIL'
        ELSE 'PASS'
    END,
    current_timestamp()
FROM silver.crm_sales_details sd
LEFT JOIN gold.dim_customer cu
    ON sd.sls_cust_id = cu.customer_id
    AND cu.is_current = true
WHERE cu.customer_key IS NULL;

In [0]:
--BUSINESS LOGIC CHECK
INSERT INTO dq.data_quality_results
SELECT
    'crm_sales_details',
    'invalid_sales_calculation',
    COUNT(*) AS failed_count,
    COUNT(*) AS total_count,
    TRY_DIVIDE(COUNT(*) , COUNT(*)) AS failure_percentage,
    CASE 
        WHEN COUNT(*) > 0 THEN 'FAIL'
        ELSE 'PASS'
    END,
    current_timestamp()
FROM silver.crm_sales_details
WHERE ABS(sls_sales - (sls_quantity * sls_price)) > 0.01;

In [0]:
--FAIL PIPELINE
SELECT *
FROM dq.data_quality_results
WHERE status = 'FAIL'
AND run_timestamp >= current_timestamp() - INTERVAL 1 HOUR;

